# Chapter 11: Using PyTorch to Fight Cancer (LUNA Grand Challenge)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/emreaslan7/ai/blob/main/notebooks/deep-learning-with-pytorch/11-using-pytorch-to-fight-cancer.ipynb)

This notebook accompanies Chapter 11 of *Deep Learning with PyTorch (2nd Edition)* by Howard Huang, Eli Stevens, Luca Antiga, and Thomas Viehmann.

### Core Concepts Implemented:
1. **CT Physics & Radiodensity:** Hounsfield Units (HU) calibrated scale and clinical windowing/leveling (Lung vs Mediastinum).
2. **Coordinate Geometry:** Affine transformations between array storage indices $(I, R, C)$ and millimeter patient world coordinates $(X, Y, Z)$.
3. **Synthetic 3D Volumetric CT Generation:** Simulating a calibrated 3D thoracic scan containing chest wall, ribs, lungs, and nodules.
4. **Three Orthogonal Planar Projections:** Axial, Coronal, and Sagittal cross-sectional slicing.
5. **Multi-Slice Nodule Morphology:** Capturing spherical growth and contraction profiles across contiguous $Z$-slices.
6. **PyTorch 3D Subvolume Ingestion:** Extracting fixed-size cubic subvolume tensors (`torch.Size([B, C, D, H, W])`) for 3D CNN architectures.

In [ ]:
import math
from collections import namedtuple
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

# Set seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch Version: {torch.__version__} | Active Device: {device}")

## 2. Coordinate Geometry: Patient World Space (XYZ) vs Array Indices (IRC)

In CT imaging:
- **World Space $(X, Y, Z)$:** Continuous floating-point coordinates measured in millimeters (mm) from the scanner isocenter.
- **Voxel Array Space $(I, R, C)$:** Discrete integers indexing the 3D tensor buffer:
  - $I$ (`Index`): Slice along $Z$-axis (axial gantry position).
  - $R$ (`Row`): Vertical position along $Y$-axis.
  - $C$ (`Column`): Horizontal position along $X$-axis.

The affine conversion formulas (assuming diagonal direction matrix) are:
$$\mathbf{x} = \mathbf{O} + \mathbf{D} \cdot (\mathbf{v} \odot \mathbf{s})$$
$$c = \text{round}\left(\frac{x - O_x}{s_x}\right), \quad r = \text{round}\left(\frac{y - O_y}{s_y}\right), \quad i = \text{round}\left(\frac{z - O_z}{s_z}\right)$$

In [ ]:
# Define strongly-typed named tuples
IrcTuple = namedtuple("IrcTuple", ["index", "row", "col"])
XyzTuple = namedtuple("XyzTuple", ["x", "y", "z"])

def irc2xyz(coord_irc: IrcTuple, origin_xyz: XyzTuple, spacing_xyz: XyzTuple) -> XyzTuple:
    """Converts array indices (I, R, C) to millimeter world coordinates (X, Y, Z)."""
    x = coord_irc.col * spacing_xyz.x + origin_xyz.x
    y = coord_irc.row * spacing_xyz.y + origin_xyz.y
    z = coord_irc.index * spacing_xyz.z + origin_xyz.z
    return XyzTuple(x=x, y=y, z=z)

def xyz2irc(coord_xyz: XyzTuple, origin_xyz: XyzTuple, spacing_xyz: XyzTuple) -> IrcTuple:
    """Converts millimeter world coordinates (X, Y, Z) to array indices (I, R, C)."""
    col = int(round((coord_xyz.x - origin_xyz.x) / spacing_xyz.x))
    row = int(round((coord_xyz.y - origin_xyz.y) / spacing_xyz.y))
    index = int(round((coord_xyz.z - origin_xyz.z) / spacing_xyz.z))
    return IrcTuple(index=index, row=row, col=col)

# Verification of bidirectional consistency
test_origin = XyzTuple(x=-180.0, y=-180.0, z=-350.0)
test_spacing = XyzTuple(x=0.75, y=0.75, z=1.5)
original_irc = IrcTuple(index=120, row=240, col=310)

computed_xyz = irc2xyz(original_irc, test_origin, test_spacing)
recovered_irc = xyz2irc(computed_xyz, test_origin, test_spacing)

print(f"Original IRC:  {original_irc}")
print(f"Computed XYZ:  X={computed_xyz.x:.2f}mm, Y={computed_xyz.y:.2f}mm, Z={computed_xyz.z:.2f}mm")
print(f"Recovered IRC: {recovered_irc}")
assert original_irc == recovered_irc, "Coordinate roundtrip check failed!"

## 3. Synthetic 3D Thoracic CT Volume Generator

To enable instant local execution without downloading multi-gigabyte LUNA tarballs, we generate a high-fidelity synthetic 3D thoracic volume ($64 \times 128 \times 128$ voxels) with authentic Hounsfield Unit calibrations:
- **Ambient Air:** $-1000\text{ HU}$
- **Chest Wall & Mediastinum (Soft Tissue):** $+40\text{ HU}$
- **Ribs & Spine (Cortical Bone):** $+750\text{ HU}$
- **Lung Cavities (Parenchyma):** $-750\text{ HU}$
- **Solitary Pulmonary Nodule:** $+60\text{ HU}$ placed inside the right lung cavity.

In [ ]:
def create_synthetic_thoracic_ct(shape=(64, 128, 128), nodule_irc=(32, 60, 42), nodule_radius_voxels=5.0):
    """
    Generates an anatomically structured 3D volume in Hounsfield Units.
    """
    nz, ny, nx = shape
    volume = np.full(shape, -1000.0, dtype=np.float32)  # Ambient air (-1000 HU)
    
    # 2D Axial Slice Grid
    y_grid, x_grid = np.ogrid[:ny, :nx]
    center_y, center_x = ny / 2.0, nx / 2.0
    
    # 2D Thorax Boundary (Soft Tissue +40 HU)
    thorax_2d = (((y_grid - center_y) / (ny * 0.42)) ** 2 + ((x_grid - center_x) / (nx * 0.44)) ** 2) <= 1.0
    
    # 2D Ribs & Outer Bone Ring (+750 HU)
    inner_boundary = (((y_grid - center_y) / (ny * 0.39)) ** 2 + ((x_grid - center_x) / (nx * 0.41)) ** 2) <= 1.0
    bone_ring_2d = thorax_2d & (~inner_boundary)
    
    # 2D Spine Vertebra (+850 HU)
    spine_center_y = center_y + (ny * 0.28)
    spine_2d = ((y_grid - spine_center_y) ** 2 + (x_grid - center_x) ** 2) <= (ny * 0.08) ** 2
    
    # 2D Right and Left Lung Cavities (-750 HU)
    right_lung_2d = (((y_grid - center_y) / (ny * 0.30)) ** 2 + ((x_grid - (center_x - nx * 0.22)) / (nx * 0.16)) ** 2) <= 1.0
    left_lung_2d  = (((y_grid - center_y) / (ny * 0.30)) ** 2 + ((x_grid - (center_x + nx * 0.22)) / (nx * 0.16)) ** 2) <= 1.0
    lungs_2d = right_lung_2d | left_lung_2d
    
    # Base 2D Axial Slice in HU
    slice_2d = np.full((ny, nx), -1000.0, dtype=np.float32)
    slice_2d[thorax_2d] = 40.0
    slice_2d[bone_ring_2d] = 750.0
    slice_2d[spine_2d] = 850.0
    slice_2d[lungs_2d] = -750.0
    
    # Broadcast 2D slice across all axial Z-slices
    volume[:] = slice_2d[np.newaxis, :, :]
    
    # 3D Solitary Spherical Pulmonary Nodule (+60 HU)
    z_grid_3d, y_grid_3d, x_grid_3d = np.ogrid[:nz, :ny, :nx]
    ni, nr, nc = nodule_irc
    dist_sq = (z_grid_3d - ni) ** 2 + (y_grid_3d - nr) ** 2 + (x_grid_3d - nc) ** 2
    nodule_mask = dist_sq <= (nodule_radius_voxels ** 2)
    volume[nodule_mask] = 60.0
    
    # Add mild Gaussian detector noise
    noise = np.random.normal(0, 15.0, size=shape).astype(np.float32)
    volume = volume + noise
    
    return torch.from_numpy(volume), nodule_irc

ct_volume, true_nodule_irc = create_synthetic_thoracic_ct()
print(f"Synthesized CT Tensor Shape: {ct_volume.shape} | Voxel Count: {ct_volume.numel()}")
print(f"HU Range: [{ct_volume.min():.1f}, {ct_volume.max():.1f}] | Nodule Center (IRC): {true_nodule_irc}")

## 4. Radiologist Windowing & Normalization

To visualize thoracic structures without display saturation, we implement standard radiologist windowing:
- **Lung Window:** $W = 1500, L = -600$ (Optimal for nodule and bronchial detail)
- **Mediastinal Window:** $W = 350, L = 40$ (Optimal for soft tissue, heart, and muscle)

$$I_{\text{display}} = \text{clip}\left(\frac{\text{HU} - (L - W/2)}{W}, 0.0, 1.0\right)$$

In [ ]:
def apply_windowing(volume: torch.Tensor, window_width: float, window_level: float) -> torch.Tensor:
    """Applies linear window-level clipping and normalizes to [0, 1]."""
    min_val = window_level - window_width / 2.0
    max_val = window_level + window_width / 2.0
    clamped = torch.clamp(volume, min=min_val, max=max_val)
    return (clamped - min_val) / (max_val - min_val)

lung_windowed = apply_windowing(ct_volume, window_width=1500.0, window_level=-600.0)
mediastinum_windowed = apply_windowing(ct_volume, window_width=350.0, window_level=40.0)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
slice_idx = true_nodule_irc[0]

axes[0].imshow(lung_windowed[slice_idx].numpy(), cmap="gray")
axes[0].set_title(f"Lung Window (W=1500, L=-600)\nSlice Index {slice_idx}")
axes[0].axis("off")

axes[1].imshow(mediastinum_windowed[slice_idx].numpy(), cmap="gray")
axes[1].set_title(f"Mediastinum Window (W=350, L=40)\nSlice Index {slice_idx}")
axes[1].axis("off")

plt.tight_layout()
plt.show()

## 5. Three Orthogonal Planar Views (Axial, Coronal, Sagittal)

Inspecting volumetric medical scans requires orthogonal re-slicing:
- **Axial Plane:** $XY$-slice at fixed $Z$ (`Index`)
- **Coronal Plane:** $XZ$-slice at fixed $Y$ (`Row`)
- **Sagittal Plane:** $YZ$-slice at fixed $X$ (`Col`)

In [ ]:
def plot_orthogonal_views(vol: torch.Tensor, center_irc: tuple, title="Orthogonal Slicing"):
    """Plots Axial, Coronal, and Sagittal cross-sections centered on center_irc."""
    ci, cr, cc = center_irc
    
    axial_slice    = vol[ci, :, :].numpy()
    coronal_slice  = vol[:, cr, :].numpy()
    sagittal_slice = vol[:, :, cc].numpy()
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # Axial (XY)
    axes[0].imshow(axial_slice, cmap="gray", origin="upper")
    axes[0].axvline(cc, color="red", linestyle="--", alpha=0.7)
    axes[0].axhline(cr, color="red", linestyle="--", alpha=0.7)
    axes[0].set_title(f"Axial Plane (Index {ci})")
    axes[0].set_xlabel("Column (X)")
    axes[0].set_ylabel("Row (Y)")
    
    # Coronal (XZ)
    axes[1].imshow(coronal_slice, cmap="gray", origin="lower")
    axes[1].axvline(cc, color="red", linestyle="--", alpha=0.7)
    axes[1].axhline(ci, color="red", linestyle="--", alpha=0.7)
    axes[1].set_title(f"Coronal Plane (Row {cr})")
    axes[1].set_xlabel("Column (X)")
    axes[1].set_ylabel("Index (Z)")
    
    # Sagittal (YZ)
    axes[2].imshow(sagittal_slice, cmap="gray", origin="lower")
    axes[2].axvline(cr, color="red", linestyle="--", alpha=0.7)
    axes[2].axhline(ci, color="red", linestyle="--", alpha=0.7)
    axes[2].set_title(f"Sagittal Plane (Col {cc})")
    axes[2].set_xlabel("Row (Y)")
    axes[2].set_ylabel("Index (Z)")
    
    plt.suptitle(title, fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

plot_orthogonal_views(lung_windowed, true_nodule_irc, title="Orthogonal Views Centered on Pulmonary Nodule")

## 6. Multi-Slice Nodule Morphology (Expansion & Contraction)

True 3D nodules exhibit spherical symmetry across contiguous $Z$-slices. Slicing through the volume illustrates the cross-sectional area expanding to its maximum diameter at the centroid slice and contracting symmetrically.

In [ ]:
ci, cr, cc = true_nodule_irc
crop_half = 12
z_offsets = [-4, -3, -2, -1, 0, 1, 2, 3, 4]

fig, axes = plt.subplots(1, len(z_offsets), figsize=(18, 3))

for idx, dz in enumerate(z_offsets):
    slice_z = ci + dz
    patch = lung_windowed[slice_z, cr - crop_half:cr + crop_half, cc - crop_half:cc + crop_half].numpy()
    axes[idx].imshow(patch, cmap="gray")
    axes[idx].set_title(f"Slice {slice_z} (dz={dz:+d})")
    axes[idx].axis("off")

plt.suptitle("Multi-Slice Profile: Nodule Appearance, Growth, and Decay", fontsize=13, y=1.05)
plt.tight_layout()
plt.show()

## 7. PyTorch 3D Subvolume Extraction & CNN Input Preparation

Finally, we implement the 3D subvolume extraction pipeline that feeds candidate patches into PyTorch 3D Convolutional Neural Networks (`nn.Conv3d`):
- **Shape:** `torch.Size([Batch, Channel, Depth, Height, Width])`
- **Padding:** Safe edge boundary handling padding with ambient air ($-1000\text{ HU}$) if the candidate lies near scan margins.

In [ ]:
def extract_candidate_subvolume(ct_vol: torch.Tensor, center_irc: IrcTuple, crop_shape=(16, 32, 32)) -> torch.Tensor:
    """
    Extracts a 3D subvolume of crop_shape (Depth, Height, Width) centered at center_irc,
    with boundary padding if necessary. Returns tensor with shape (1, 1, D, H, W).
    """
    half_d = crop_shape[0] // 2
    half_h = crop_shape[1] // 2
    half_w = crop_shape[2] // 2
    
    start_i = max(0, center_irc.index - half_d)
    end_i   = min(ct_vol.shape[0], center_irc.index + half_d)
    start_r = max(0, center_irc.row - half_h)
    end_r   = min(ct_vol.shape[1], center_irc.row + half_h)
    start_c = max(0, center_irc.col - half_w)
    end_c   = min(ct_vol.shape[2], center_irc.col + half_w)
    
    subvolume = ct_vol[start_i:end_i, start_r:end_r, start_c:end_c]
    
    # Pad if near boundaries
    subvolume_padded = F.pad(
        subvolume,
        (
            max(0, half_w - (center_irc.col - start_c)),
            max(0, half_w - (end_c - center_irc.col)),
            max(0, half_h - (center_irc.row - start_r)),
            max(0, half_h - (end_r - center_irc.row)),
            max(0, half_d - (center_irc.index - start_i)),
            max(0, half_d - (end_i - center_irc.index))
        ),
        value=-1000.0
    )
    
    # Format as (B=1, C=1, D, H, W)
    return subvolume_padded.unsqueeze(0).unsqueeze(0)

# Extract subvolume centered at the true nodule
target_crop_shape = (16, 32, 32)
candidate_tensor = extract_candidate_subvolume(ct_volume, IrcTuple(*true_nodule_irc), crop_shape=target_crop_shape)

print(f"Extracted PyTorch 3D Batch Tensor Shape: {candidate_tensor.shape}")
assert candidate_tensor.shape == (1, 1, 16, 32, 32), "Unexpected tensor shape!"

# Verification with PyTorch 3D Convolution layer
conv3d = nn.Conv3d(in_channels=1, out_channels=16, kernel_size=3, padding=1)
output_feature_map = conv3d(candidate_tensor)
print(f"Conv3d Output Feature Map Shape: {output_feature_map.shape}")
print("✅ Pipeline verified! Ready for Chapter 12 dataset ingestion and Chapter 13 3D classifier!")